In [1]:
from __future__ import annotations

import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv

from fraud_intelligence.explainability.shap_explainer import (
    build_tree_explainer,
    calculate_global_feature_importance,
    calculate_shap_values,
    get_raw_feature_names,
    transform_features_for_explanation,
    validate_feature_frame,
)
from fraud_intelligence.explainability.stability import (
    evaluate_shap_stability,
    summarise_shap_stability,
)
from fraud_intelligence.models.data_contract import load_modelling_data
from fraud_intelligence.models.model_loading import (
    load_frozen_xgboost_components,
)

RANDOM_SEED = 42
BACKGROUND_SAMPLE_SIZE = 500
GLOBAL_EXPLANATION_SAMPLE_SIZE = 2_000
LOCAL_TOP_FEATURE_COUNT = 10
STABILITY_BACKGROUND_SEEDS = [42, 52, 62]
STABILITY_TOP_FEATURE_COUNT = 10

NOTEBOOK_WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in (
            NOTEBOOK_WORKING_DIRECTORY,
            *NOTEBOOK_WORKING_DIRECTORY.parents,
        )
        if (
            (candidate_directory / "configs").is_dir()
            and (candidate_directory / "scripts").is_dir()
            and (candidate_directory / "src").is_dir()
        )
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root. Expected an ancestor folder "
        "containing configs/, scripts/, and src/."
    )
print(f"Repository root detected: {PROJECT_ROOT}")

import os
os.chdir(PROJECT_ROOT)
print(f"Python working directory set to: {Path.cwd()}")

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

FIGURE_DIRECTORY = PROJECT_ROOT / "reports" / "figures"
TABLE_DIRECTORY = PROJECT_ROOT / "reports" / "tables"

FIGURE_DIRECTORY.mkdir(parents=True, exist_ok=True)
TABLE_DIRECTORY.mkdir(parents=True, exist_ok=True)

phase5_script_path = PROJECT_ROOT / "scripts" / "evaluate_phase5_final_holdout.py"

phase5_spec = importlib.util.spec_from_file_location(
    "phase5_final_holdout",
    phase5_script_path,
)

if phase5_spec is None or phase5_spec.loader is None:
    raise ImportError(
        "Could not load scripts/evaluate_phase5_final_holdout.py."
    )

phase5_module = importlib.util.module_from_spec(phase5_spec)
phase5_spec.loader.exec_module(phase5_module)

if not hasattr(phase5_module, "load_final_test_data"):
    raise AttributeError(
        "evaluate_phase5_final_holdout.py does not expose "
        "load_final_test_data()."
    )

with (PROJECT_ROOT / "configs" / "modeling.yaml").open(
    encoding="utf-8"
) as config_file:
    modelling_config = yaml.safe_load(config_file)

print("Loading frozen Phase 5 champion model...")
frozen_components = load_frozen_xgboost_components()

print("Loading leakage-safe chronological train data...")
modelling_data = load_modelling_data()

print("Loading untouched final holdout data...")
X_test, y_test, test_timestamps = phase5_module.load_final_test_data(
    modelling_config
)

if len(X_test) != len(y_test):
    raise ValueError("Final holdout features and labels have different lengths.")

if len(X_test) == 0:
    raise ValueError("The final holdout dataset is empty.")

raw_feature_names = get_raw_feature_names(frozen_components.preprocessor)

X_train = validate_feature_frame(
    feature_frame=modelling_data.X_train,
    expected_input_features=raw_feature_names,
)

X_test = validate_feature_frame(
    feature_frame=X_test,
    expected_input_features=raw_feature_names,
)

background_sample_size = min(BACKGROUND_SAMPLE_SIZE, len(X_train))

background_raw_features = X_train.sample(
    n=background_sample_size,
    random_state=RANDOM_SEED,
)

background_transformed_features, transformed_feature_names = (
    transform_features_for_explanation(
        preprocessor=frozen_components.preprocessor,
        raw_feature_frame=background_raw_features,
    )
)

print("Scoring final holdout with frozen XGBoost classifier...")
test_transformed_features, test_feature_names = (
    transform_features_for_explanation(
        preprocessor=frozen_components.preprocessor,
        raw_feature_frame=X_test,
    )
)

if transformed_feature_names != test_feature_names:
    raise ValueError(
        "Training-background and holdout transformed feature names differ."
    )

raw_classifier_probabilities = (
    frozen_components.classifier.predict_proba(
        test_transformed_features
    )[:, 1]
)

if not np.isfinite(raw_classifier_probabilities).all():
    raise ValueError("The frozen classifier returned non-finite probabilities.")

if not (
    (raw_classifier_probabilities >= 0.0)
    & (raw_classifier_probabilities <= 1.0)
).all():
    raise ValueError(
        "The frozen classifier returned probabilities outside [0, 1]."
    )

rng = np.random.default_rng(RANDOM_SEED)

global_sample_size = min(
    GLOBAL_EXPLANATION_SAMPLE_SIZE,
    len(X_test),
)

global_positions = np.sort(
    rng.choice(
        len(X_test),
        size=global_sample_size,
        replace=False,
    )
)

X_global = X_test.iloc[global_positions].copy()

global_transformed_features, global_feature_names = (
    transform_features_for_explanation(
        preprocessor=frozen_components.preprocessor,
        raw_feature_frame=X_global,
    )
)

if global_feature_names != transformed_feature_names:
    raise ValueError(
        "Global explanation sample feature names differ from background names."
    )

print(
    f"Calculating global SHAP values for {len(X_global):,} "
    "deterministically sampled holdout transactions..."
)

global_explainer = build_tree_explainer(
    classifier=frozen_components.classifier,
    background_features=background_transformed_features,
)

global_explanation = calculate_shap_values(
    explainer=global_explainer,
    transformed_features=global_transformed_features,
    feature_names=global_feature_names,
)

global_importance = calculate_global_feature_importance(
    explanation_result=global_explanation,
)

global_importance.insert(
    0,
    "rank",
    np.arange(1, len(global_importance) + 1),
)

global_importance.to_csv(
    TABLE_DIRECTORY / "phase7a_global_shap_importance.csv",
    index=False,
)

top_global_importance = global_importance.head(20).copy()

plt.figure(figsize=(11, 8))
plt.barh(
    top_global_importance["transformed_feature"][::-1],
    top_global_importance["mean_absolute_shap_value"][::-1],
    color="#2563EB",
)
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Transformed model feature")
plt.title(
    "Phase 7A: Global SHAP Feature Importance\n"
    "Frozen Phase 5 XGBoost Champion — Final Holdout Sample"
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIRECTORY / "phase7a_global_shap_importance.png",
    dpi=200,
    bbox_inches="tight",
)
plt.close()

fraud_positions = np.flatnonzero(y_test.to_numpy() == 1)
legitimate_positions = np.flatnonzero(y_test.to_numpy() == 0)

if len(fraud_positions) == 0 or len(legitimate_positions) == 0:
    raise ValueError(
        "Representative case selection requires both fraud and legitimate "
        "transactions in the final holdout."
    )

highest_scoring_fraud_position = fraud_positions[
    np.argmax(raw_classifier_probabilities[fraud_positions])
]

lowest_scoring_fraud_position = fraud_positions[
    np.argmin(raw_classifier_probabilities[fraud_positions])
]

lowest_scoring_legitimate_position = legitimate_positions[
    np.argmin(raw_classifier_probabilities[legitimate_positions])
]

highest_scoring_legitimate_position = legitimate_positions[
    np.argmax(raw_classifier_probabilities[legitimate_positions])
]

representative_case_positions = {
    "correctly_ranked_fraud": int(highest_scoring_fraud_position),
    "correctly_ranked_legitimate": int(lowest_scoring_legitimate_position),
    "missed_fraud_low_score": int(lowest_scoring_fraud_position),
    "unnecessary_review_high_score": int(
        highest_scoring_legitimate_position
    ),
}

case_rows = []

for case_group, row_position in representative_case_positions.items():
    case_rows.append(
        {
            "case_group": case_group,
            "holdout_row_position": row_position,
            "true_label": int(y_test.iloc[row_position]),
            "raw_classifier_probability": float(
                raw_classifier_probabilities[row_position]
            ),
            "transaction_timestamp": float(test_timestamps[row_position]),
        }
    )

representative_case_table = pd.DataFrame(case_rows)

representative_case_table.to_csv(
    TABLE_DIRECTORY / "phase7a_representative_cases.csv",
    index=False,
)

X_representative = X_test.iloc[
    representative_case_table["holdout_row_position"].to_numpy()
].copy()

representative_transformed_features, representative_feature_names = (
    transform_features_for_explanation(
        preprocessor=frozen_components.preprocessor,
        raw_feature_frame=X_representative,
    )
)

if representative_feature_names != transformed_feature_names:
    raise ValueError(
        "Representative-case feature names differ from background names."
    )

representative_explanation = calculate_shap_values(
    explainer=global_explainer,
    transformed_features=representative_transformed_features,
    feature_names=representative_feature_names,
)

local_contributor_tables = []

for local_row_index, case_row in representative_case_table.iterrows():
    case_group = case_row["case_group"]
    shap_values = representative_explanation.shap_values[local_row_index]

    contributor_table = pd.DataFrame(
        {
            "case_group": case_group,
            "holdout_row_position": int(
                case_row["holdout_row_position"]
            ),
            "true_label": int(case_row["true_label"]),
            "raw_classifier_probability": float(
                case_row["raw_classifier_probability"]
            ),
            "transformed_feature": representative_feature_names,
            "shap_value": shap_values,
            "absolute_shap_value": np.abs(shap_values),
        }
    )

    top_contributors = (
        contributor_table.sort_values(
            "absolute_shap_value",
            ascending=False,
        )
        .head(LOCAL_TOP_FEATURE_COUNT)
        .copy()
    )

    local_contributor_tables.append(top_contributors)

    plot_table = top_contributors.sort_values(
        "shap_value",
        ascending=True,
    )

    bar_colours = np.where(
        plot_table["shap_value"] > 0,
        "#DC2626",
        "#2563EB",
    )

    plt.figure(figsize=(11, 6))
    plt.barh(
        plot_table["transformed_feature"],
        plot_table["shap_value"],
        color=bar_colours,
    )
    plt.axvline(0.0, color="black", linewidth=0.8)
    plt.xlabel("SHAP value on XGBoost raw-margin scale")
    plt.ylabel("Transformed model feature")
    plt.title(
        f"Phase 7A Local SHAP Contributors: {case_group}\n"
        f"True label={int(case_row['true_label'])}; "
        f"raw XGBoost score={case_row['raw_classifier_probability']:.4f}"
    )
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIRECTORY / f"phase7a_local_shap_{case_group}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

local_contributors = pd.concat(
    local_contributor_tables,
    ignore_index=True,
)

local_contributors.to_csv(
    TABLE_DIRECTORY / "phase7a_local_shap_contributors.csv",
    index=False,
)

print(
    "Calculating SHAP stability across deterministic training-background "
    "samples..."
)

stability_shap_values_by_seed: dict[int, np.ndarray] = {}

for background_seed in STABILITY_BACKGROUND_SEEDS:
    stability_background_raw = X_train.sample(
        n=background_sample_size,
        random_state=background_seed,
    )

    stability_background_transformed, stability_feature_names = (
        transform_features_for_explanation(
            preprocessor=frozen_components.preprocessor,
            raw_feature_frame=stability_background_raw,
        )
    )

    if stability_feature_names != representative_feature_names:
        raise ValueError(
            "Stability background feature names differ from representative "
            "case feature names."
        )

    stability_explainer = build_tree_explainer(
        classifier=frozen_components.classifier,
        background_features=stability_background_transformed,
    )

    stability_explanation = calculate_shap_values(
        explainer=stability_explainer,
        transformed_features=representative_transformed_features,
        feature_names=representative_feature_names,
    )

    stability_shap_values_by_seed[background_seed] = (
        stability_explanation.shap_values
    )

stability_pairwise = evaluate_shap_stability(
    shap_values_by_seed=stability_shap_values_by_seed,
    feature_names=representative_feature_names,
    case_group_names=representative_case_table["case_group"].tolist(),
    top_feature_count=STABILITY_TOP_FEATURE_COUNT,
)

stability_summary = summarise_shap_stability(
    pairwise_stability_table=stability_pairwise,
)

stability_pairwise.to_csv(
    TABLE_DIRECTORY / "phase7a_shap_stability_pairwise.csv",
    index=False,
)

stability_summary.to_csv(
    TABLE_DIRECTORY / "phase7a_shap_stability_summary.csv",
    index=False,
)

analysis_metadata = pd.DataFrame(
    [
        {
            "model_name": "xgboost",
            "model_version": frozen_components.model_version,
            "mlflow_run_id": frozen_components.mlflow_run_id,
            "preprocessor_model_uri": (
                frozen_components.preprocessor_model_uri
            ),
            "classifier_model_uri": frozen_components.classifier_model_uri,
            "calibration_method": "sigmoid",
            "background_dataset": "chronological training split only",
            "background_sample_size": background_sample_size,
            "background_random_seed": RANDOM_SEED,
            "global_explanation_dataset": (
                "deterministic random final-holdout sample"
            ),
            "global_explanation_sample_size": global_sample_size,
            "global_explanation_random_seed": RANDOM_SEED,
            "local_case_count": len(representative_case_table),
            "stability_background_seeds": (
                " | ".join(map(str, STABILITY_BACKGROUND_SEEDS))
            ),
            "stability_top_feature_count": STABILITY_TOP_FEATURE_COUNT,
            "shap_output_scale": "XGBoost raw margin",
            "interpretation_limit": (
                "SHAP values describe model-feature contributions, not "
                "causal evidence or proof of fraud."
            ),
        }
    ]
)

analysis_metadata.to_csv(
    TABLE_DIRECTORY / "phase7a_explainability_metadata.csv",
    index=False,
)

print("\n=== PHASE 7A SHAP ANALYSIS COMPLETE ===")
print(f"Champion model: {frozen_components.model_version}")
print(f"MLflow run ID: {frozen_components.mlflow_run_id}")
print(f"Global SHAP sample rows: {global_sample_size:,}")
print(f"Background training rows: {background_sample_size:,}")
print(f"Representative cases: {len(representative_case_table)}")
print("\nRepresentative cases:")
display(representative_case_table)

print("\nTop 10 global SHAP contributors:")
display(global_importance.head(10))

print("\nSHAP stability summary:")
display(stability_summary)

print("\nSaved figures:")
for figure_path in sorted(FIGURE_DIRECTORY.glob("phase7a_*.png")):
    print(figure_path.relative_to(PROJECT_ROOT))

print("\nSaved tables:")
for table_path in sorted(TABLE_DIRECTORY.glob("phase7a_*.csv")):
    print(table_path.relative_to(PROJECT_ROOT))

Repository root detected: C:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Financial Fraud Risk & Investigation Intelligence Platform
Python working directory set to: C:\Users\Softmgmt\Desktop\MAIN_FOLDER\WHOLE_DATA\ML_JOBS\MACHINE_LEARNING_PROJECTS\Financial Fraud Risk & Investigation Intelligence Platform
Loading frozen Phase 5 champion model...


Loading leakage-safe chronological train data...
Loading untouched final holdout data...
Scoring final holdout with frozen XGBoost classifier...
Calculating global SHAP values for 2,000 deterministically sampled holdout transactions...


100%|===================| 1994/2000 [02:41<00:00]        

Calculating SHAP stability across deterministic training-background samples...

=== PHASE 7A SHAP ANALYSIS COMPLETE ===
Champion model: xgboost-v1.0.0
MLflow run ID: a1f99cff34cf44728d3b6c9499ffeee3
Global SHAP sample rows: 2,000
Background training rows: 500
Representative cases: 4

Representative cases:


,case_group,holdout_row_position,true_label,raw_classifier_probability,transaction_timestamp
0,correctly_ranked_fraud,33672,1,0.999690,14096183.0
1,correctly_ranked_legitimate,37502,0,0.000211,14229044.0
2,missed_fraud_low_score,3563,1,0.004672,13227323.0
3,unnecessary_review_high_score,80686,0,0.999426,15557752.0



Top 10 global SHAP contributors:


,rank,transformed_feature,mean_absolute_shap_value
0,1,amount_global_24h_zscore,0.169216
1,2,C1,0.169206
2,3,amount_global_168h_zscore,0.158346
3,4,C13,0.155574
4,5,card1,0.148949
5,6,C5,0.130318
6,7,V70,0.130127
7,8,D1,0.129612
8,9,card6_target_encoded,0.127404
9,10,C14,0.126739



SHAP stability summary:


,case_group,seed_pair_count,mean_jaccard_similarity,min_jaccard_similarity,max_jaccard_similarity,mean_shared_top_feature_count
0,correctly_ranked_fraud,3,1.000000,1.000000,1.0,10.000000
1,correctly_ranked_legitimate,3,1.000000,1.000000,1.0,10.000000
2,missed_fraud_low_score,3,0.878788,0.818182,1.0,9.333333
3,unnecessary_review_high_score,3,0.878788,0.818182,1.0,9.333333



Saved figures:
reports\figures\phase7a_global_shap_importance.png
reports\figures\phase7a_local_shap_correctly_ranked_fraud.png
reports\figures\phase7a_local_shap_correctly_ranked_legitimate.png
reports\figures\phase7a_local_shap_missed_fraud_low_score.png
reports\figures\phase7a_local_shap_unnecessary_review_high_score.png

Saved tables:
reports\tables\phase7a_explainability_metadata.csv
reports\tables\phase7a_global_shap_importance.csv
reports\tables\phase7a_local_shap_contributors.csv
reports\tables\phase7a_representative_cases.csv
reports\tables\phase7a_shap_stability_pairwise.csv
reports\tables\phase7a_shap_stability_summary.csv


In [ ]:
from pathlib import Path

import pandas as pd

REPORT_DIRECTORY = PROJECT_ROOT / "reports" / "evaluation"
REPORT_DIRECTORY.mkdir(parents=True, exist_ok=True)

global_importance_path = (
    TABLE_DIRECTORY / "phase7a_global_shap_importance.csv"
)
representative_cases_path = (
    TABLE_DIRECTORY / "phase7a_representative_cases.csv"
)
local_contributors_path = (
    TABLE_DIRECTORY / "phase7a_local_shap_contributors.csv"
)
stability_pairwise_path = (
    TABLE_DIRECTORY / "phase7a_shap_stability_pairwise.csv"
)
stability_summary_path = (
    TABLE_DIRECTORY / "phase7a_shap_stability_summary.csv"
)
metadata_path = (
    TABLE_DIRECTORY / "phase7a_explainability_metadata.csv"
)

global_importance = pd.read_csv(global_importance_path)
representative_cases = pd.read_csv(representative_cases_path)
local_contributors = pd.read_csv(local_contributors_path)
stability_pairwise = pd.read_csv(stability_pairwise_path)
stability_summary = pd.read_csv(stability_summary_path)
analysis_metadata = pd.read_csv(metadata_path)

metadata = analysis_metadata.iloc[0].to_dict()


def markdown_table(
    table: pd.DataFrame,
    columns: list[str],
    decimal_columns: list[str] | None = None,
) -> str:
    """Create a Markdown table without requiring optional packages."""
    decimal_columns = decimal_columns or []

    display_table = table.loc[:, columns].copy()

    for column in decimal_columns:
        if column in display_table.columns:
            display_table[column] = display_table[column].map(
                lambda value: f"{float(value):.6f}"
            )

    display_table = display_table.fillna("")

    for column in display_table.columns:
        display_table[column] = (
            display_table[column]
            .astype(str)
            .str.replace("|", "\\|", regex=False)
        )

    header = "| " + " | ".join(display_table.columns) + " |"
    separator = "| " + " | ".join(
        ["---"] * len(display_table.columns)
    ) + " |"

    rows = [
        "| " + " | ".join(row) + " |"
        for row in display_table.astype(str).values.tolist()
    ]

    return "\n".join([header, separator, *rows])


top_global_features = global_importance.head(10).copy()

representative_case_display = representative_cases.copy()
representative_case_display["raw_classifier_probability"] = (
    representative_case_display["raw_classifier_probability"].map(
        lambda value: f"{float(value):.6f}"
    )
)
representative_case_display["transaction_timestamp"] = (
    representative_case_display["transaction_timestamp"].map(
        lambda value: f"{float(value):.0f}"
    )
)

stability_display = stability_summary.copy()
for stability_column in [
    "mean_jaccard_similarity",
    "min_jaccard_similarity",
    "max_jaccard_similarity",
    "mean_shared_top_feature_count",
]:
    stability_display[stability_column] = stability_display[
        stability_column
    ].map(lambda value: f"{float(value):.6f}")

case_interpretations = {
    "correctly_ranked_fraud": (
        "A fraud-labelled transaction with the highest raw XGBoost risk "
        "score among final-holdout fraud cases. This is a representative "
        "correct high-risk ranking, not proof that any contributor caused fraud."
    ),
    "correctly_ranked_legitimate": (
        "A legitimate transaction with the lowest raw XGBoost risk score "
        "among final-holdout legitimate cases. This is a representative "
        "correct low-risk ranking."
    ),
    "missed_fraud_low_score": (
        "A fraud-labelled transaction with the lowest raw XGBoost risk "
        "score among final-holdout fraud cases. It illustrates an important "
        "missed-fraud / low-priority error pattern for later Phase 7B analysis."
    ),
    "unnecessary_review_high_score": (
        "A legitimate transaction with the highest raw XGBoost risk score "
        "among final-holdout legitimate cases. It illustrates an unnecessary-"
        "review / false-positive error pattern for later Phase 7B analysis."
    ),
}

local_case_sections: list[str] = []

for _, case_row in representative_cases.iterrows():
    case_group = str(case_row["case_group"])

    contributor_subset = (
        local_contributors.loc[
            local_contributors["case_group"] == case_group
        ]
        .sort_values(
            "absolute_shap_value",
            ascending=False,
        )
        .head(5)
        .copy()
    )

    contributor_subset["shap_value"] = contributor_subset["shap_value"].map(
        lambda value: f"{float(value):+.6f}"
    )
    contributor_subset["absolute_shap_value"] = contributor_subset[
        "absolute_shap_value"
    ].map(lambda value: f"{float(value):.6f}")

    local_case_sections.append(
        f"""### {case_group.replace("_", " ").title()}

{case_interpretations.get(case_group, "Representative final-holdout case.")}

- Holdout row position: `{int(case_row["holdout_row_position"])}`
- True label: `{int(case_row["true_label"])}`
- Raw frozen-XGBoost probability: `{float(case_row["raw_classifier_probability"]):.6f}`
- Transaction timestamp value: `{float(case_row["transaction_timestamp"]):.0f}`

Top absolute SHAP contributors:

{markdown_table(
    contributor_subset,
    columns=[
        "transformed_feature",
        "shap_value",
        "absolute_shap_value",
    ],
)}
"""
    )

report_content = f"""# Phase 7A — Explainability Report

## Purpose

This report documents global and local SHAP explanations for the locked fraud-risk
champion model. The analysis evaluates which transformed input features contributed
to the model's predictions, illustrates representative correct and incorrect
ranking outcomes, and checks local attribution stability when the SHAP background
reference sample changes.

SHAP values describe how features contributed to this model's prediction. They do
not prove fraud, establish causation, or demonstrate that a transaction attribute
independently caused risk.

## Frozen Analysis Scope

| Item | Value |
| --- | --- |
| Model name | `{metadata["model_name"]}` |
| Model version | `{metadata["model_version"]}` |
| MLflow training run | `{metadata["mlflow_run_id"]}` |
| Preprocessor artifact | `{metadata["preprocessor_model_uri"]}` |
| Classifier artifact | `{metadata["classifier_model_uri"]}` |
| Probability calibration selected in Phase 5 | `{metadata["calibration_method"]}` |
| SHAP output scale | `{metadata["shap_output_scale"]}` |
| SHAP background data | `{metadata["background_dataset"]}` |
| Background sample size | `{int(metadata["background_sample_size"]):,}` |
| Global explanation sample | `{metadata["global_explanation_dataset"]}` |
| Global explanation sample size | `{int(metadata["global_explanation_sample_size"]):,}` |
| Local representative cases | `{int(metadata["local_case_count"])}` |
| Stability background seeds | `{metadata["stability_background_seeds"]}` |
| Stability top-feature count | `{int(metadata["stability_top_feature_count"])}` |

The XGBoost classifier and its fitted preprocessing pipeline were loaded from the
frozen Phase 5 artifacts. No model fitting, feature fitting, threshold selection,
calibration fitting, or policy optimisation occurred in this notebook.

The SHAP background sample was selected deterministically from chronological
training data only. The global explanation sample and representative explanations
use the final holdout after the model family, features, calibration method, and
decision policy had already been locked.

## Global Feature Contributions

Global importance is calculated as the mean absolute SHAP value across a
deterministic sample of `{int(metadata["global_explanation_sample_size"]):,}`
final-holdout transactions. A higher value means that the transformed feature
moved the frozen model's raw prediction more on average within this explained
sample. It does not imply that the feature caused fraud.

![Global SHAP importance](../figures/phase7a_global_shap_importance.png)

### Top 10 Global Contributors

{markdown_table(
    top_global_features,
    columns=[
        "rank",
        "transformed_feature",
        "mean_absolute_shap_value",
    ],
    decimal_columns=["mean_absolute_shap_value"],
)}

The full ranking is saved in:

```text
reports/tables/phase7a_global_shap_importance.csv
```

## Representative Local Explanations

The four cases below are selected to show both correct ranking behaviour and
important error patterns. The displayed risk values are raw frozen-XGBoost
probabilities for explanation context. Phase 5's sigmoid calibration remains the
approved probability-calibration approach for operational policy evaluation.

{markdown_table(
    representative_case_display,
    columns=[
        "case_group",
        "holdout_row_position",
        "true_label",
        "raw_classifier_probability",
        "transaction_timestamp",
    ],
)}

{"".join(local_case_sections)}

Local explanation figures are stored in:

```text
reports/figures/phase7a_local_shap_correctly_ranked_fraud.png
reports/figures/phase7a_local_shap_correctly_ranked_legitimate.png
reports/figures/phase7a_local_shap_missed_fraud_low_score.png
reports/figures/phase7a_local_shap_unnecessary_review_high_score.png
```

## Attribution Stability

Stability was evaluated by recalculating local SHAP values for the same four
representative cases using deterministic training-background samples with seeds
`{metadata["stability_background_seeds"]}`. For each pair of background samples,
the top `{int(metadata["stability_top_feature_count"])}` features ranked by
absolute SHAP contribution were compared using Jaccard similarity.

A Jaccard similarity of `1.0` means the two runs selected the same top-feature
set. Lower values indicate that the leading transformed contributors varied when
the background reference sample changed.

{markdown_table(
    stability_display,
    columns=[
        "case_group",
        "seed_pair_count",
        "mean_jaccard_similarity",
        "min_jaccard_similarity",
        "max_jaccard_similarity",
        "mean_shared_top_feature_count",
    ],
)}

The complete pairwise results are stored in:

```text
reports/tables/phase7a_shap_stability_pairwise.csv
reports/tables/phase7a_shap_stability_summary.csv
```

## Limitations

- SHAP explains the fitted model's behaviour, not causal fraud mechanisms.
- The global ranking uses a deterministic final-holdout sample rather than every
  holdout transaction to keep the analysis computationally practical.
- Feature names are transformed preprocessor outputs; categorical variables may
  appear as one-hot-encoded levels rather than as a single raw business feature.
- SHAP values were calculated on the XGBoost raw-margin scale because this is the
  additive scale supported by the interventional TreeExplainer configuration.
- Representative cases illustrate model ranking behaviour and error patterns;
  they are not estimates of the prevalence of each error type.
- Attribution stability tests sensitivity to the selected SHAP background
  reference sample only. It does not demonstrate causal stability, fairness,
  temporal robustness, or generalisation beyond the public IEEE-CIS benchmark.
- All conclusions are benchmark findings under the project’s documented
  assumptions and must not be represented as results from a real financial
  institution.

## Output Inventory

```text
reports/figures/phase7a_global_shap_importance.png
reports/figures/phase7a_local_shap_correctly_ranked_fraud.png
reports/figures/phase7a_local_shap_correctly_ranked_legitimate.png
reports/figures/phase7a_local_shap_missed_fraud_low_score.png
reports/figures/phase7a_local_shap_unnecessary_review_high_score.png
reports/tables/phase7a_explainability_metadata.csv
reports/tables/phase7a_global_shap_importance.csv
reports/tables/phase7a_local_shap_contributors.csv
reports/tables/phase7a_representative_cases.csv
reports/tables/phase7a_shap_stability_pairwise.csv
reports/tables/phase7a_shap_stability_summary.csv
```
"""

report_path = REPORT_DIRECTORY / "explainability_report.md"
report_path.write_text(report_content, encoding="utf-8")

print("=== PHASE 7A REPORT CREATED ===")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Report size: {report_path.stat().st_size:,} bytes")
print(
    "Global top feature: "
    f"{global_importance.iloc[0]['transformed_feature']}"
)
print(
    "Mean stability range: "
    f"{stability_summary['mean_jaccard_similarity'].min():.6f} to "
    f"{stability_summary['mean_jaccard_similarity'].max():.6f}"
)